**Colab-ready:** reads the immutable corrected dataset from Google Drive, verifies it, and writes a JSON report back to Drive. This notebook does not train models.

# 01 — Independent dataset verification

**Research question for this stage:** Can we confirm that the **accepted corrected** count matrices and metadata are internally consistent before designing any classifier?

**Inputs:** `data/validated/ibd_dataset_sample_aligned.zip`, `configs/dataset.json`.
**Output:** a new timestamped JSON report under `runs/data_validation/`.

**What this does NOT do:** re-create source sequencing data, normalise counts, remove genes, fit a feature selector, or validate any previous experimental result. The dataset is read-only.

## A. Load the canonical manifest and independently written validator

The manifest pins the accepted dataset SHA-256 and expected cohort properties. A different file with the same name will be rejected.

In [ ]:
from pathlib import Path
import os
import sys

# Colab runs notebooks with /content as its working directory, not the notebook's Drive folder.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')  # Explicit Google authorization, never unattended

# An environment override supports a clone, a shared drive, or another folder name.
candidates = []
if os.environ.get('CMPS355_ROOT'):
    candidates.append(Path(os.environ['CMPS355_ROOT']).expanduser())
if IN_COLAB:
    candidates += [Path('/content/drive/MyDrive/CMPS355_Research'),
                   Path('/content/CMPS355_Research')]
candidates += [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'CMPS355_Research']
ROOT = next((p.resolve() for p in candidates if (p / '.cmps355_root').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        'Project root not found. On Colab, upload/extract CMPS355_Research under '
        'MyDrive, or set os.environ["CMPS355_ROOT"] to the actual project folder. '
        'On local Jupyter, launch it from the project root.'
    )
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Runtime:', 'Google Colab' if IN_COLAB else 'Local Jupyter/Python')
print('Project:', ROOT)

import json
from cmps355.data_validation import validate_archive, save_validation_report
manifest = json.loads((ROOT / "configs/dataset.json").read_text(encoding="utf-8"))
dataset_path = ROOT / manifest["artifact"]
print("Canonical input:", dataset_path.relative_to(ROOT))


### Install/verify scientific dependencies

Only NumPy and pandas are required for Stage 1. Colab has its own Jupyter frontend; local JupyterLab and R are **not** required here. Pinned versions support consistency across runs. If Colab requests a runtime restart after changing installed packages, restart and rerun from the beginning.

In [ ]:
# Install only when the *installed* scientific package versions differ from the tested ones.
# This also works on an ordinary local Jupyter runtime (network access if installation is needed).
from importlib.metadata import version, PackageNotFoundError
import subprocess

expected = {'numpy': '2.3.5', 'pandas': '2.2.3'}
found = {}
for package in expected:
    try:
        found[package] = version(package)
    except PackageNotFoundError:
        found[package] = None
needs_install = any(found[p] != expected[p] for p in expected)
if needs_install:
    print('Version mismatch:', found, '->', expected)
    print('Installing stage-one dependencies. If imports were already loaded, restart the runtime afterward.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)
else:
    print('Scientific dependencies already match tested versions:', found)


## B. Run scientific integrity checks

The validator checks the raw CSV headers (before pandas can rename duplicates), exact count/metadata ID equality and ordering, ID uniqueness across all four cohorts, class labels, source BioProjects, numeric finite nonnegative integer counts, nonzero sample totals and shared gene names. It does not assume that rows are aligned merely because both tables have the same number of rows.

In [ ]:
from time import perf_counter
start_time = perf_counter()
result = validate_archive(dataset_path, manifest)
print("Integrity checks:", result["status"])
print("Elapsed seconds:", round(perf_counter() - start_time, 2))
print("Unique samples:", result["unique_sample_count"])
print("Shared genes:", result["common_gene_count"])
print("Source BioProjects:", result["n_source_bioprojects"])

## C. Inspect class imbalance and the supplied count distributions

The raw supplied gene-count totals are **not sequencing depth measurements**, and cohort differences do not automatically imply biological effects. We display their distributions so the future preprocessing protocol can be justified rather than assumed.

In [ ]:
import pandas as pd
cohort_rows = []
for cohort in result["cohorts"]:
    cohort_rows.append({
        "cohort": cohort["cohort"],
        "patients": cohort["samples"],
        "CD": cohort["labels"].get("CD", 0),
        "Not IBD": cohort["labels"].get("Not IBD", 0),
        "CD fraction": round(cohort["labels"].get("CD", 0) / cohort["samples"], 3),
        "gene columns": cohort["genes"],
        "median assigned gene counts": int(cohort["median_assigned_counts"]),
        "fraction of zero entries": round(cohort["fraction_zero_gene_counts"], 3),
    })
display(pd.DataFrame(cohort_rows))
print("Class totals:", result["class_totals"])
print("BioProject memberships:")
for cohort in result["cohorts"]:
    print(f"  {cohort['cohort']}: {cohort['source_bioprojects']}")

## D. Decide whether the accepted data is ready for the *next* design stage

A PASS means **internal integrity of the accepted archive**: 689 unique samples, aligned IDs, no invalid count entries, and 9,983 shared gene names. It does not prove the original source RData matches every count, nor that cohorts are batch-confounding-free. Do not perform modelling or pathway analysis until Stage 2 defines suitable methods and preprocessing.

In [ ]:
report_path = save_validation_report(result, ROOT)
print("Saved:", report_path.relative_to(ROOT))
assert result["status"] == "PASS"
assert result["unique_sample_count"] == 689
assert len(result["cohorts"]) == 4
assert result["common_gene_count"] == 9983
assert result["n_source_bioprojects"] == 6
print("STAGE 1 DATA VERIFICATION: PASS")
print("Next: Stage 2 — read original papers, verify official software, and build independent reference tests.")